In [1]:
import os, time, subprocess, psutil
import polars as pl
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import HTML, display

# ════════════════════════════════════════════════════════════════
# CONFIG — edit here
# ════════════════════════════════════════════════════════════════
SEND_EMAIL       = True   # True = send + attach file via Outlook
DISPLAY_NOTEBOOK = True
OVERRIDE_DATE    = None    # None = auto D-1 | "2026-06-01" = manual override

_home    = os.path.expanduser("~").replace("\\", "/")
ATD_PATH = (
    f"{_home}/Concentrix Corporation/"
    "WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/ATD_Final.parquet"
)

# ── Email recipients ────────────────────────────────────────────
EMAIL_TO = (
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com"
)
 
EMAIL_CC = (
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "van.tran@concentrix.com;"
    "duonghoangvu.pham@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "ExpediaVN_Training_Team@concentrix.com;"
    "ExpediaVN_QA_Team@concentrix.com;"
)

# ── LOB mapping & filter ────────────────────────────────────────
LOB_MAP = {
    "Support_LG_Nesting":"Lodging","Lodging_Nesting":"Lodging",
    "Support_NL_Nesting":"Non_Lodging","Non_Lodging_Nesting":"Non_Lodging",
    "LG Chat CSG Training":"Training","LG Chat CSG":"Lodging",
    "Support_Flex_SME":"SME",
}
KEEP_LOBS = ["Lodging","Non_Lodging"]    # SME excluded globally

# ── Highlight thresholds (unit: hours) ─────────────────────────
THR_MISS = 1.0    # Missed Productive > 1h     → highlight red
THR_LATE = 0.25   # Login Late / Early Leave > 15 min → highlight red
THR_OVR  = 0.25   # Overbreak / Overlunch > 15 min   → highlight red
THR_UNS  = 0.5    # Unscheduled > 30 min              → highlight red

# ── Row filter ─────────────────────────────────────────────────
MIN_TOTAL = 1.0   # Rows with Total ≤ this value (hrs) are excluded from all tables

print("✓ Config loaded")

✓ Config loaded


In [2]:
print("📂 Loading ATD_Final.parquet...")
atd_raw = pl.read_parquet(ATD_PATH)
print(f"✓ {atd_raw.shape[0]:,} rows | {len(atd_raw.columns)} cols")

atd = (
    atd_raw
    .with_columns(pl.col("LOB").replace(LOB_MAP).alias("LOB"))
    .filter(pl.col("LOB").is_in(KEEP_LOBS))
)

# Exclude Termination shifts (agents who have left / inactive rows)
rows_pre_term = atd.shape[0]
atd = atd.filter(
    pl.col("Original.Shift").is_not_null() &
    (pl.col("Original.Shift") != "Termination")
)
print(f"✓ Termination filter: {rows_pre_term:,} → {atd.shape[0]:,} rows "
      f"({rows_pre_term - atd.shape[0]:,} Termination rows removed)")

# ── Date setup ──────────────────────────────────────────────────
_now      = datetime.now()
days_back = 2 if _now.hour < 6 else 1   # before 06:00 use D-2, else D-1
if OVERRIDE_DATE:
    _rd = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
    print(f"⚠️  OVERRIDE: {OVERRIDE_DATE}")
else:
    _rd = _now - timedelta(days=days_back)

report_date_d  = _rd.date()
report_date_s  = _rd.strftime("%d-%b-%Y")
month_start    = _rd.replace(day=1).date()
current_month  = _rd.strftime("%b-%y")
date_7d_start  = report_date_d - timedelta(days=6)
_pm            = pd.Period(report_date_d, "M")
last_4_months  = [(_pm - i).strftime("%b-%y") for i in range(3,-1,-1)]
EMAIL_SUBJECT  = f"Expedia VN – Missed Productive Report – {report_date_s}"

# ── EXCLUDE FUTURE DATES ────────────────────────────────────────
# The roster is published for the full week, so future dates already have
# Target values but no actual SUM Productive → inflates Missed Productive.
# Global filter: keep only Date ≤ report_date_d (respects OVERRIDE_DATE).
rows_before = atd.shape[0]
atd         = atd.filter(pl.col("Date") <= report_date_d)
rows_after  = atd.shape[0]
dropped     = rows_before - rows_after
print(f"✓ Future-date filter: {rows_before:,} → {rows_after:,} rows "
      f"({dropped:,} future rows removed, cutoff ≤ {report_date_d})")

print(f"✓ D-1   : {report_date_d}  ({report_date_s})")
print(f"✓ Month : {current_month}  |  start: {month_start}")
print(f"✓ 7-Day : {date_7d_start} → {report_date_d}")
print(f"✓ MoM   : {last_4_months}")

📂 Loading ATD_Final.parquet...
✓ 38,164 rows | 62 cols
✓ Termination filter: 36,640 → 31,443 rows (5,197 Termination rows removed)
✓ Future-date filter: 31,443 → 30,694 rows (749 future rows removed, cutoff ≤ 2026-08-28)
✓ D-1   : 2026-08-28  (28-Aug-2026)
✓ Month : Aug-26  |  start: 2026-08-01
✓ 7-Day : 2026-08-22 → 2026-08-28
✓ MoM   : ['May-26', 'Jun-26', 'Jul-26', 'Aug-26']


In [3]:
# ════════════════════════════════════════════════════════════════
# COMPUTE FUNCTIONS
# ════════════════════════════════════════════════════════════════

METRIC_KEYS = ["Missed","Late","Leave","O.Break","O.Lunch","Unsched"]

def _hrs_to_hhmm(v):
    """Convert decimal hours to HH:MM string."""
    try:
        f = float(v)
        if pd.isna(f) or f < 0: return "00:00"
        t = round(f * 60)
        return f"{t // 60:02d}:{t % 60:02d}"
    except:
        return "\u2014"

def _safe(frame, col, fallback=0.0):
    """Return a Polars expression for col, or a literal fallback if col is missing."""
    if col in frame.columns:
        return pl.col(col).cast(pl.Float64, strict=False).fill_null(fallback)
    return pl.lit(fallback)

def prep_frame(frame):
    """Add 6 standardized metric columns (unit: hours)."""
    idle_src = ["Training_Idle [Sum]","Coaching_Idle [Sum]","Orther_Status [Sum]"]
    avail    = [c for c in idle_src if c in frame.columns]
    idle_sum = (
        pl.sum_horizontal([pl.col(c).cast(pl.Float64,strict=False).fill_null(0) for c in avail])
        if avail else pl.lit(0.0)
    )
    # Unscheduled = (Training_Idle + Coaching_Idle + Other_Status) - Training (scheduled)
    # Clamp to 0: if scheduled training covers all idle time, result = 0
    unsched = (idle_sum - _safe(frame, "Training")).clip(lower_bound=0.0)
    return frame.with_columns([
        _safe(frame,"Measure_missed_hours").alias("Missed"),
        _safe(frame,"Time_Late").alias("Late"),
        _safe(frame,"Time_Leave").alias("Leave"),
        _safe(frame,"Measure_Over_Break").alias("O.Break"),
        _safe(frame,"Measure_Over_Lunch").alias("O.Lunch"),
        unsched.alias("Unsched"),
    ])

def _agg_by(frame, group_cols):
    """Group by group_cols and sum all METRIC_KEYS."""
    return prep_frame(frame).group_by(group_cols).agg([pl.col(k).sum() for k in METRIC_KEYS])

def _make_gt_row(df, id_col, sum_cols):
    """Build Grand Total row (2 dp) from data rows only (already filtered)."""
    gt = {c:"" for c in df.columns}
    gt[id_col] = "Grand Total"; gt["_row_type"] = "total"
    for c in sum_cols:
        try:
            gt[c] = round(pd.to_numeric(df.loc[df["_row_type"]=="data", c], errors="coerce").sum(), 2)
        except: pass
    return gt

def _add_gt_row(df, id_col="Supervisor Name"):
    """Filter rows by MIN_TOTAL then append Grand Total row (column-wise sum)."""
    df = df.copy()
    present = [k for k in METRIC_KEYS if k in df.columns]
    df["_tmp"]      = df[present].sum(axis=1).round(2)
    df["_row_type"] = "data"
    df = df[df["_tmp"] > MIN_TOTAL].drop(columns=["_tmp"]).reset_index(drop=True)
    if df.empty: return df
    gt = {c: "" for c in df.columns}
    gt[id_col]      = "Grand Total"
    gt["_row_type"] = "total"
    for c in present:
        gt[c] = round(pd.to_numeric(df[c], errors="coerce").sum(), 2)
    return pd.concat([df, pd.DataFrame([gt])], ignore_index=True)
def compute_t1_mom():
    """Monthly Missed Productive per supervisor (last 4 months)."""
    frame = atd.filter(pl.col("Month").is_in(last_4_months))
    if frame.is_empty(): return pd.DataFrame(), []
    agg = (
        prep_frame(frame).group_by(["Month","Supervisor Name"])
        .agg(pl.col("Missed").sum()).to_pandas()
    )
    months_f = [m for m in last_4_months if m in agg["Month"].values]
    sups     = sorted(agg["Supervisor Name"].dropna().unique())

    rows = []
    for sup in sups:
        row = {"Supervisor Name": sup, "_row_type": "data"}; gt = 0.0
        for m in months_f:
            sub = agg[(agg["Supervisor Name"]==sup) & (agg["Month"]==m)]
            v   = float(sub["Missed"].iloc[0]) if len(sub) and pd.notna(sub["Missed"].iloc[0]) else 0.0
            row[m] = round(v,2); gt += v
        row["Grand Total"] = round(gt,2)
        rows.append(row)

    df = pd.DataFrame(rows)
    # Remove rows with Grand Total <= MIN_TOTAL
    df = df[pd.to_numeric(df["Grand Total"], errors="coerce").fillna(0) > MIN_TOTAL].reset_index(drop=True)
    df["_row_type"] = "data"
    if df.empty: return df, months_f
    gt_row = _make_gt_row(df, "Supervisor Name", months_f + ["Grand Total"])
    return pd.concat([df, pd.DataFrame([gt_row])], ignore_index=True), months_f

# ── TABLE 2: Supervisor × 7-Day daily — Missed Productive only ────────
def compute_t2_7days_daily():
    """Daily Missed Productive per supervisor over the last 7 days."""
    frame = atd.filter((pl.col("Date") >= date_7d_start) & (pl.col("Date") <= report_date_d))
    if frame.is_empty(): return pd.DataFrame(), []
    agg = (
        prep_frame(frame).group_by(["Date","Supervisor Name"])
        .agg(pl.col("Missed").sum()).to_pandas()
    )
    agg["D"] = pd.to_datetime(agg["Date"]).dt.strftime("%d-%b")
    d_range  = [report_date_d - timedelta(days=i) for i in range(6,-1,-1)]
    d_strs   = [d.strftime("%d-%b") for d in d_range]
    found_d  = [d for d in d_strs if d in agg["D"].values]
    sups     = sorted(agg["Supervisor Name"].dropna().unique())

    rows = []
    for sup in sups:
        row = {"Supervisor Name": sup, "_row_type": "data"}; total = 0.0
        for ds in found_d:
            sub = agg[(agg["Supervisor Name"]==sup) & (agg["D"]==ds)]
            v   = float(sub["Missed"].iloc[0]) if len(sub) and pd.notna(sub["Missed"].iloc[0]) else 0.0
            row[ds] = round(v,2); total += v
        row["Total"] = round(total,2)
        rows.append(row)

    df = pd.DataFrame(rows)
    # Remove rows with Total <= MIN_TOTAL
    df = df[pd.to_numeric(df["Total"], errors="coerce").fillna(0) > MIN_TOTAL].reset_index(drop=True)
    df["_row_type"] = "data"
    if df.empty: return df, found_d
    gt_row = _make_gt_row(df, "Supervisor Name", found_d + ["Total"])
    return pd.concat([df, pd.DataFrame([gt_row])], ignore_index=True), found_d

# ── TABLE 3: Supervisor × D-1 ─────────────────────────────────────────
def compute_t3_d1():
    """All metrics per supervisor for D-1. Rows with Total ≤ MIN_TOTAL excluded."""
    frame = atd.filter(pl.col("Date") == report_date_d)
    if frame.is_empty(): return pd.DataFrame()
    agg = _agg_by(frame, ["Supervisor Name"]).sort("Supervisor Name").to_pandas()
    return _add_gt_row(agg)

# ── TABLE 4: Supervisor × Current Month MTD ──────────────────────────
def compute_t4_month():
    """All metrics per supervisor MTD. Rows with Total ≤ MIN_TOTAL excluded."""
    frame = atd.filter(
        (pl.col("Month")==current_month) & (pl.col("Date")>=month_start) & (pl.col("Date")<=report_date_d)
    )
    if frame.is_empty(): return pd.DataFrame()
    agg = _agg_by(frame, ["Supervisor Name"]).sort("Supervisor Name").to_pandas()
    return _add_gt_row(agg)

# ── TABLE 5: Top 5 Agents by Missed Productive (MTD) ─────────────────
def compute_t5_top5():
    """Top 5 agents ranked by Missed Productive (MTD). Rows with Total ≤ MIN_TOTAL excluded."""
    frame = atd.filter(
        (pl.col("Month")==current_month) & (pl.col("Date")>=month_start) & (pl.col("Date")<=report_date_d)
    )
    if frame.is_empty(): return pd.DataFrame()
    agg = _agg_by(frame, ["OracleID","Employee Name","Supervisor Name"]).sort("Missed",descending=True).head(5).to_pandas()
    agg["_tmp"]      = agg[[k for k in METRIC_KEYS if k in agg.columns]].sum(axis=1).round(2)
    agg["_row_type"] = "data"
    agg = agg[agg["_tmp"] > MIN_TOTAL].drop(columns=["_tmp"]).reset_index(drop=True)
    if agg.empty: return agg
    gt = {"OracleID": "", "Employee Name": "Grand Total", "Supervisor Name": "", "_row_type": "total"}
    for k in METRIC_KEYS:
        if k in agg.columns:
            gt[k] = round(pd.to_numeric(agg[k], errors="coerce").sum(), 2)
    return pd.concat([agg, pd.DataFrame([gt])], ignore_index=True)

# ── ATTACHMENT: agent-level detail for current month MTD ─────────────
def prepare_attachment():
    """
    Build a temporary Excel (missed_hours_detail.xlsx) with MTD agent-level data.
    Written to the system temp folder; deleted automatically after email is sent.
    """
    # Full current month up to report_date_d
    frame = atd.filter(
        (pl.col("Month") == current_month) &
        (pl.col("Date") >= month_start) &
        (pl.col("Date") <= report_date_d)
    )
    if frame.is_empty(): return pd.DataFrame(), None

    detail_pl = prep_frame(frame)
    # Format: (source_col_in_ATD_Final, display_name_in_Excel)
    # Computed cols (Missed/Late/Leave/O.Break/O.Lunch/Unsched) are added by prep_frame()
    SRC = [
        ("Date",             "Date"),
        ("OracleID",         "OracleID"),
        ("Employee Name",    "Employee Name"),
        ("Supervisor Name",  "Supervisor Name"),
        ("LOB",              "LOB"),
        ("Email Id",         "Email Id"),
        ("Original.Shift",   "Shift"),
        ("Start Time",       "Start"),
        ("End Time",         "End"),
        ("Duration",         "Duration"),
        ("Target",           "Target"),
        ("SUM Productive",   "SUM Productive"),
        ("Lateness",         "Lateness"),
        ("Lunch [Sum]",      "Lunch (hrs)"),
        ("Break [Sum]",      "Break (hrs)"),
        ("Time_Late",        "_tl_hrs"),    # converted to hh:mm below
        ("Time_Leave",       "_tv_hrs"),    # converted to hh:mm below
        ("Missed",           "Missed (hrs)"),
        ("Late",             "Late (hrs)"),
        ("Leave",            "Leave (hrs)"),
        ("O.Break",          "O.Break (hrs)"),
        ("O.Lunch",          "O.Lunch (hrs)"),
        ("Unsched",          "Unsched (hrs)"),
    ]
    avail_exprs = [pl.col(sc).alias(dn) for sc,dn in SRC if sc in detail_pl.columns]
    df = detail_pl.select(avail_exprs).to_pandas()

    # Convert Time_Late and Time_Leave to hh:mm
    for raw,col_name in [("_tl_hrs","Time_Late (hh:mm)"),("_tv_hrs","Time_Leave (hh:mm)")]:
        if raw in df.columns:
            df[col_name] = df[raw].apply(_hrs_to_hhmm)
            df.drop(columns=[raw], inplace=True)

    # Convert all hour metric columns to hh:mm format
    METRIC_HRS = {
        "Lunch (hrs)":   "Lunch (hh:mm)",
        "Break (hrs)":   "Break (hh:mm)",
        "Missed (hrs)":  "Missed Productive (hh:mm)",
        "Late (hrs)":    "Login Late (hh:mm)",
        "Leave (hrs)":   "Early Leave (hh:mm)",
        "O.Break (hrs)": "Overbreak (hh:mm)",
        "O.Lunch (hrs)": "Overlunch (hh:mm)",
        "Unsched (hrs)": "Unscheduled (hh:mm)",
    }
    for src_col, dst_col in METRIC_HRS.items():
        if src_col in df.columns:
            df[dst_col] = df[src_col].apply(_hrs_to_hhmm)
            df.drop(columns=[src_col], inplace=True)

    sort_cols = [c for c in ["Supervisor Name","Employee Name"] if c in df.columns]
    if sort_cols: df = df.sort_values(sort_cols).reset_index(drop=True)

    # Write to system temp dir — file deleted after email send, nothing stored permanently
    import tempfile
    att_path = os.path.join(tempfile.gettempdir(), "missed_hours_detail.xlsx")
    try:
        df.to_excel(att_path, index=False, sheet_name="Detail")
        print(f"✓ Temp file ready: missed_hours_detail.xlsx ({len(df):,} rows, {current_month} MTD)")
    except Exception as e:
        print(f"❌ Could not create temp file: {e}")
        return df, None
    return df, att_path

print("✓ Compute functions defined")

✓ Compute functions defined


In [4]:
# ════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ════════════════════════════════════════════════════════════════
HDR_DARK = "#1a3a5c"; HDR_MID  = "#1f5c99"; HDR_LITE = "#2e86c1"
BLU_ROW  = "#dce8f5"; WHT_ROW  = "#ffffff"
TOT_BG   = "#1a3a5c"; YLW_HDR  = "#FFD700"
MISS_BG  = "#fde8ea"; MISS_FG  = "#9b1c2a"
FONT     = "font-family:Arial,sans-serif;font-size:11px;"
TH_S     = (FONT + "padding:5px 8px;color:#fff;font-weight:bold;"
            "white-space:nowrap;text-align:center;border:1px solid rgba(255,255,255,0.2);")
TD_S     = FONT + "padding:4px 8px;border:1px solid #dce8f5;white-space:nowrap;"
TD_TOT   = (FONT + "padding:4px 8px;white-space:nowrap;background:" + "#1a3a5c" +
            ";color:#fff;font-weight:bold;border:1px solid rgba(255,255,255,0.12);")

CSS = (
    "body{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}"
    ".t{border-collapse:collapse;font-size:11px;font-family:Arial,sans-serif;white-space:nowrap;width:auto}"
    ".t thead th{padding:5px 8px;color:#fff;font-weight:bold;text-align:center;border:1px solid rgba(255,255,255,0.2)}"
    ".t tbody td{padding:4px 8px;border:1px solid #dce8f5;text-align:left}"
    ".t tbody tr.blu td{background:#dce8f5}"
    ".t tbody tr.wht td{background:#ffffff}"
    ".t tbody tr.tot td{background:#1a3a5c!important;color:#fff!important;font-weight:bold!important}"
    ".miss{background:#fde8ea!important;color:#9b1c2a!important;font-weight:bold!important}"
)

# ── Column display name mapping ──────────────────────────────
COL_DISPLAY = {
    "Missed":  "Missed Productive",
    "Late":    "Login Late",
    "Leave":   "Early Leave",
    "O.Break": "Overbreak",
    "O.Lunch": "Overlunch",
    "Unsched": "Unscheduled",
    "Total":   "Total",
}

# ════════════════════════════════════════════════════════════════
# FORMAT HELPERS
# ════════════════════════════════════════════════════════════════
def fv(v):
    if v is None or (isinstance(v,float) and pd.isna(v)): return "\u2014"
    try: return f"{float(v):.2f}"
    except: return str(v)

def _tstr(v):
    if v is None or (isinstance(v,float) and pd.isna(v)): return "\u2014"
    return str(v)

def _miss_style(val, thr, for_email):
    try:
        f = float(val)
        if not pd.isna(f) and f > thr:
            return ("background:" + MISS_BG + ";color:" + MISS_FG + ";font-weight:bold;"
                    if for_email else "miss")
    except: pass
    return ""

def _td(val, bg_r, thr, for_email, right=True):
    v  = fv(val); al = "right" if right else "left"
    ms = _miss_style(val, thr, for_email)
    if for_email:
        sty = ms if ms else ("background:" + bg_r + ";")
        return "<td style=\"" + TD_S + sty + "text-align:" + al + ";\">" + v + "</td>"
    cls = (" class=\"" + ms + "\"") if ms else ""
    sty = "" if ms else ("background:" + bg_r + ";")
    return "<td" + cls + " style=\"" + TD_S + sty + "text-align:" + al + ";\">" + v + "</td>"

def _td_tot(val, for_email, right=True, dark=False):
    v = fv(val); al = "right" if right else "left"
    xb = "background:#0d2e4d;" if dark else ""
    return "<td style=\"" + TD_TOT + xb + "text-align:" + al + ";\">" + v + "</td>"

# ════════════════════════════════════════════════════════════════
# SECTION HEADERS
# ════════════════════════════════════════════════════════════════
def _grp_hdr(title, subtitle, color, for_email=False):
    ts  = FONT + "font-size:14px;font-weight:bold;color:#fff;margin:0;"
    ss  = FONT + "font-size:11px;color:#fff;margin:3px 0 0;"
    inn = "<p style=\"" + ts + "\">" + title + "</p><p style=\"" + ss + "\">" + subtitle + "</p>"
    if for_email:
        return ('<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:18px 0 8px;">'
                '<tr><td style="background:' + color + ';padding:10px 14px;border-radius:4px;">'
                + inn + '</td></tr></table>')
    return '<div style="background:' + color + ';padding:10px 14px;border-radius:4px;margin:18px 0 8px;">' + inn + '</div>'

def _sec_hdr(badge, note, color=HDR_MID, for_email=False):
    bs = ("background:" + YLW_HDR + ";color:#000;font-weight:bold;font-size:12px;"
          "padding:3px 10px;border-radius:3px;")
    ns = (FONT + "font-size:10.5px;color:#555;background:#f8f8f8;border-left:3px solid "
          + color + ";padding:4px 10px;margin:4px 0 6px;border-radius:0 3px 3px 0;")
    if for_email:
        return ('<p style="margin:16px 0 2px;"><span style="' + bs + '">' + badge + '</span></p>'
                + '<p style="' + ns + '">' + note + '</p>')
    return ('<div style="margin:16px 0 2px;"><span style="' + bs + '">' + badge + '</span></div>'
            + '<div style="' + ns + '">' + note + '</div>')

def _legend(for_email=False):
    sp = ("background:" + MISS_BG + ";color:" + MISS_FG + ";font-weight:bold;"
          "padding:2px 8px;margin-right:6px;border-radius:3px;font-size:10px;")
    txt = ('<span style="' + sp + '">\u25a0 &gt; threshold</span>'
           + " Missed&gt;" + str(THR_MISS) + "h &nbsp;|&nbsp; Late/Leave&gt;" + str(THR_LATE)
           + "h &nbsp;|&nbsp; O.Break/Lunch&gt;" + str(THR_OVR)
           + "h &nbsp;|&nbsp; Unscheduled = Training_Idle + Coaching_Idle + Other_Status")
    ns = FONT + "font-size:10px;color:#666;margin:4px 0 12px;"
    return ('<p style="' + ns + '">' + txt + '</p>' if for_email
            else '<div style="' + ns + '">' + txt + '</div>')

# ════════════════════════════════════════════════════════════════
# PIVOT TABLE RENDERER — Tables 1 & 2 (only Missed column per period)
# ════════════════════════════════════════════════════════════════
def _render_pivot(df, data_cols, gt_col, id_col="Supervisor Name", for_email=False):
    """
    Pivot: id_col | data_col_1 (Missed hrs) | ... | gt_col (Grand Total / Total)
    Used for Table 1 (MoM) and Table 2 (7-day daily).
    """
    if df is None or df.empty or not data_cols:
        return '<p style="color:#888;">\u26a0\ufe0f No data available.</p>'
    t_cls = "" if for_email else 'class="t" '
    h = ['<table ' + t_cls + 'style="border-collapse:collapse;width:auto;' + FONT + '"><thead><tr>']
    h.append('<th style="' + TH_S + 'background:' + HDR_DARK + ';min-width:150px;">' + id_col + '</th>')
    for dc in data_cols:
        h.append('<th style="' + TH_S + 'background:' + HDR_MID + ';">' + dc
                 + '<br><span style="font-size:9px;font-weight:normal;">Missed (hrs)</span></th>')
    h.append('<th style="' + TH_S + 'background:' + HDR_DARK + ';">' + gt_col + '</th>')
    h.append("</tr></thead><tbody>")

    alt = 0
    for _, row in df.iterrows():
        rt = row.get("_row_type","data"); tot = rt=="total"
        if tot:
            h.append('<tr>' if for_email else '<tr class="tot">')
        else:
            even = alt % 2 == 0; alt += 1
            bg_r = BLU_ROW if even else WHT_ROW
            h.append('<tr>' if for_email else ('<tr class="' + ("blu" if even else "wht") + '">'))

        sup = _tstr(row.get(id_col,""))
        h.append(_td_tot(sup,for_email,right=False) if tot
                 else '<td style="' + TD_S + 'background:' + bg_r + ';">' + sup + '</td>')

        for dc in data_cols:
            val = row.get(dc, 0.0)
            h.append(_td_tot(val,for_email) if tot else _td(val,bg_r,THR_MISS,for_email))

        gt_val = row.get(gt_col, 0.0)
        h.append(_td_tot(gt_val,for_email,dark=True) if tot
                 else '<td style="' + TD_S + 'background:#c8d8ee;text-align:right;font-weight:bold;">'
                 + fv(gt_val) + '</td>')
        h.append("</tr>")

    h.append("</tbody></table>")
    return "".join(h)

# ════════════════════════════════════════════════════════════════
# STD TABLE RENDERER — Tables 3 & 4
# Columns: Supervisor | Missed | Late | Leave | O.Break | O.Lunch | Unsched | Total
# No Grand Total row
# ════════════════════════════════════════════════════════════════
_STD = [
    ("Supervisor Name", HDR_DARK, None,   True),
    ("Missed",          HDR_MID,  "MISS", False),
    ("Late",            HDR_MID,  "LATE", False),
    ("Leave",           HDR_MID,  "LATE", False),
    ("O.Break",         HDR_MID,  "OVR",  False),
    ("O.Lunch",         HDR_MID,  "OVR",  False),
    ("Unsched",         HDR_MID,  "UNS",  False),
]

def _thr(code):
    return {"MISS":THR_MISS,"LATE":THR_LATE,"OVR":THR_OVR,"UNS":THR_UNS}.get(code, 9999)

def _render_std(df, for_email=False):
    if df is None or df.empty:
        return '<p style="color:#888;">\u26a0\ufe0f No data for this period.</p>'
    t_cls = "" if for_email else 'class="t" '
    h = ['<table ' + t_cls + 'style="border-collapse:collapse;width:auto;' + FONT + '"><thead><tr>']
    for lbl,bg,tc,_ in _STD:
        h.append('<th style="' + TH_S + 'background:' + bg + ';">' + COL_DISPLAY.get(lbl,lbl) + '</th>')
    h.append("</tr></thead><tbody>")

    alt = 0
    for _, row in df.iterrows():
        rt = row.get("_row_type","data"); tot = rt=="total"
        if tot:  # Grand Total row (if present)
            h.append('<tr>' if for_email else '<tr class="tot">')
            for lbl,_,tc,is_txt in _STD:
                val = row.get(lbl,""); v = _tstr(val) if is_txt else fv(val)
                al  = "left" if is_txt else "right"
                h.append('<td style="' + TD_TOT + 'text-align:' + al + ';">' + v + '</td>')
            h.append("</tr>"); continue

        even = alt % 2 == 0; alt += 1; bg_r = BLU_ROW if even else WHT_ROW
        h.append('<tr>' if for_email else ('<tr class="' + ("blu" if even else "wht") + '">'))
        for lbl,bg,tc,is_txt in _STD:
            val = row.get(lbl,"")
            if is_txt:
                h.append('<td style="' + TD_S + 'background:' + bg_r + ';">' + _tstr(val) + '</td>')
            else:
                # Total column: highlight if > sum of thresholds (optional), or just format
                if lbl == "Total":
                    h.append('<td style="' + TD_S + 'background:' + bg_r
                             + ';text-align:right;font-weight:bold;">' + fv(val) + '</td>')
                else:
                    h.append(_td(val, bg_r, _thr(tc), for_email))
        h.append("</tr>")
    h.append("</tbody></table>")
    return "".join(h)

# ════════════════════════════════════════════════════════════════
# TOP 5 TABLE — Table 5 (no GT row)
# ════════════════════════════════════════════════════════════════
_TOP5 = [
    ("OracleID",        HDR_DARK, None,   True),
    ("Employee Name",   HDR_DARK, None,   True),
    ("Supervisor Name", HDR_DARK, None,   True),
    ("Missed",          HDR_MID,  "MISS", False),
    ("Late",            HDR_MID,  "LATE", False),
    ("Leave",           HDR_MID,  "LATE", False),
    ("O.Break",         HDR_MID,  "OVR",  False),
    ("O.Lunch",         HDR_MID,  "OVR",  False),
    ("Unsched",         HDR_MID,  "UNS",  False),
]

def _render_top5(df, for_email=False):
    if df is None or df.empty:
        return '<p style="color:#888;">\u26a0\ufe0f No Top-5 data.</p>'
    t_cls = "" if for_email else 'class="t" '
    h = ['<table ' + t_cls + 'style="border-collapse:collapse;width:auto;' + FONT + '"><thead><tr>']
    for lbl,bg,tc,_ in _TOP5:
        h.append('<th style="' + TH_S + 'background:' + bg + ';">' + COL_DISPLAY.get(lbl,lbl) + '</th>')
    h.append("</tr></thead><tbody>")

    alt = 0
    for _, row in df.iterrows():
        rt = row.get("_row_type","data"); tot = rt=="total"
        if tot:
            h.append('<tr>' if for_email else '<tr class="tot">')
            for lbl,_,tc,is_txt in _TOP5:
                val = row.get(lbl,""); v = _tstr(val) if is_txt else fv(val)
                al  = "left" if is_txt else "right"
                h.append('<td style="' + TD_TOT + 'text-align:' + al + ';">' + v + '</td>')
            h.append("</tr>"); continue

        even = alt % 2 == 0; alt += 1; bg_r = BLU_ROW if even else WHT_ROW
        h.append('<tr>' if for_email else ('<tr class="' + ("blu" if even else "wht") + '">'))
        for lbl,bg,tc,is_txt in _TOP5:
            val = row.get(lbl,"")
            if is_txt:
                h.append('<td style="' + TD_S + 'background:' + bg_r + ';">' + _tstr(val) + '</td>')
            elif lbl == "Total":
                h.append('<td style="' + TD_S + 'background:' + bg_r
                         + ';text-align:right;font-weight:bold;">' + fv(val) + '</td>')
            else:
                h.append(_td(val, bg_r, _thr(tc), for_email))
        h.append("</tr>")
    h.append("</tbody></table>")
    return "".join(h)

# ════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ════════════════════════════════════════════════════════════════
def build_all(for_email=False):
    P = []
    P.append(_grp_hdr(
        "\U0001f4c9 Missed Productive Report \u2014 " + report_date_s,
        "LOBs: " + ", ".join(KEEP_LOBS) + " | Source: ATD_Final.parquet | "
        + "Generated: " + datetime.now().strftime('%Y-%m-%d %H:%M'),
        "#1a3a5c", for_email
    ))

    # TABLE 1: MoM
    P.append(_sec_hdr(
        "\U0001f4c5 TABLE 1 \u2014 Supervisor Wise | Missed Productive Month-on-Month (Last 4 Months)",
        "Period: " + last_4_months[0] + " \u2192 " + last_4_months[-1]
        + " | Missed Productive (hrs) aggregated per month. Rows with Total ≤ 1h are excluded.",
        HDR_DARK, for_email
    ))
    P.append(_render_pivot(df_t1, months_t1, "Grand Total", for_email=for_email))
    P.append(_legend(for_email))

    # TABLE 2: 7-day daily
    P.append(_sec_hdr(
        "\U0001f5d3\ufe0f TABLE 2 \u2014 Supervisor Wise | Missed Productive by Day (Last 7 Days)",
        "Period: " + date_7d_start.strftime('%d-%b-%Y') + " \u2192 " + report_date_s
        + " | One column per day. Grand Total = 7-day sum. Rows with Total ≤ 1h excluded.",
        HDR_MID, for_email
    ))
    P.append(_render_pivot(df_t2, dates_t2, "Total", for_email=for_email))
    P.append(_legend(for_email))

    # TABLE 3: D-1
    P.append(_sec_hdr(
        "\U0001f4c6 TABLE 3 \u2014 Supervisor Wise | D-1 : " + report_date_s,
        "Missed Productive + Late + Leave + Over Break/Lunch + Unscheduled ng\xe0y D-1. "
        "C\u1ed9t Total = t\u1ed5ng 6 metrics.",
        "#5c1f99", for_email
    ))
    P.append(_render_std(df_t3, for_email))
    P.append(_legend(for_email))

    # TABLE 4: MTD
    P.append(_sec_hdr(
        "\U0001f4ca TABLE 4 \u2014 Supervisor Wise | Month-to-Date : " + current_month,
        "Period: " + month_start.strftime('%d-%b-%Y') + " \u2192 " + report_date_s
        + " (MTD). Total = sum of 6 metrics. Rows with Total ≤ 1h excluded.",
        "#1f7a4d", for_email
    ))
    P.append(_render_std(df_t4, for_email))
    P.append(_legend(for_email))

    # TABLE 5: Top 5
    P.append(_sec_hdr(
        "\U0001f3c6 TABLE 5 \u2014 Top 5 Agents by Missed Productive | " + current_month + " MTD",
        "Ranked descending by Missed Productive (hrs) | "
        + month_start.strftime('%d-%b-%Y') + " \u2192 " + report_date_s,
        "#8b3a00", for_email
    ))
    P.append(_render_top5(df_t5, for_email))
    P.append(_legend(for_email))
    # ── Signature ──────────────────────────────────────────────
    s1 = FONT + "font-size:11px;color:#555;margin:0 0 2px;"
    s2 = FONT + "font-size:12px;font-weight:bold;margin:0 0 2px;"
    P.append(
        '<hr style="border:none;border-top:1px solid #e0e0e0;margin:20px 0 10px;">'
        + '<p style="' + s1 + '">Thanks &amp; Regards,</p>'
        + '<p style="' + s2 + '">Chinh Nguyen</p>'
        + '<p style="' + s1 + '">Analyst, WFM Real Time Management</p>'
        + '<p style="' + s1 + 'line-height:1.6;margin-top:4px;">'
          'Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street,'
          '<br>Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam</p>'
        + '<p style="' + s1 + '">'
          'Ph: +84 986 473 419 &nbsp;|&nbsp; '
          '<a href="mailto:huuchinh.nguyen@concentrix.com" '
          'style="color:' + HDR_MID + ';font-weight:bold;text-decoration:none;">'
          'huuchinh.nguyen@concentrix.com</a></p>')
    return "".join(P)

def _greeting():
    s = FONT + "font-size:12px;margin:0 0 10px;line-height:1.7"
    return (
        '<p style="' + s + '">Dear Team,</p>'
        '<p style="' + s + '">Please find below the <strong>Missed Productive Report</strong> '
        'as of <strong>' + report_date_s + '</strong>. '
        'This report covers missed productive hours, login lateness, early leave, '
        'over-break/over-lunch, and unscheduled activities per Supervisor and Agent. '
        'A detailed agent-level data file is attached for reference.</p>'
        '<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 14px;">'
    )

def _signature():
    s  = FONT + "font-size:12px;margin:0 0 4px;line-height:1.7"
    s2 = FONT + "font-size:11px;color:#555;margin:0 0 2px;"
    return (
        '<hr style="border:none;border-top:1px solid #e0e0e0;margin:14px 0 10px;">'
        '<p style="' + s + '">Thanks &amp; Regards,</p>'
        '<p style="' + FONT + 'font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>'
        '<p style="' + s2 + '">Analyst, WFM Real Time Management</p><br>'
        '<p style="' + s2 + 'line-height:1.6">Level 4, Tower 1, OneHub Saigon, Lot C1-2, '
        'D1 Street, Saigon Hi Tech Park,<br>Tan Phu Ward, District 9, HCMC, Vietnam</p>'
        '<p style="' + s2 + '">Ph: +84 986 473 419 &nbsp;|&nbsp; '
        '<a href="mailto:huuchinh.nguyen@concentrix.com" '
        'style="color:' + HDR_MID + ';font-weight:bold;text-decoration:none;">'
        'huuchinh.nguyen@concentrix.com</a></p>'
        '<p style="' + FONT + 'font-size:10px;color:#aaa;margin-top:10px;">'
        'Generated: ' + datetime.now().strftime("%Y-%m-%d %H:%M") + ' | Source: ATD_Final.parquet</p>'
    )

# ════════════════════════════════════════════════════════════════
# COMPUTE + ATTACHMENT + DISPLAY + EMAIL
# ════════════════════════════════════════════════════════════════
print("\u23f3 Computing tables...")
df_t1, months_t1 = compute_t1_mom()
df_t2, dates_t2  = compute_t2_7days_daily()
df_t3            = compute_t3_d1()
df_t4            = compute_t4_month()
df_t5            = compute_t5_top5()
print(f"\u2713 T1 MoM     : {len(df_t1)} rows | months: {months_t1}")
print(f"\u2713 T2 7-Day   : {len(df_t2)} rows | dates : {dates_t2}")
print(f"\u2713 T3 D-1     : {len(df_t3)} rows")
print(f"\u2713 T4 MTD     : {len(df_t4)} rows")
print(f"\u2713 T5 Top-5   : {len(df_t5)} rows")

print("\u23f3 Preparing attachment...")
df_att, att_path = prepare_attachment()
print(f"\u2713 Attachment : {len(df_att)} rows | {att_path}")

# ── Display ──────────────────────────────────────────────────────────
if DISPLAY_NOTEBOOK:
    nb_html = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        "<style>" + CSS + "</style></head><body>"
        + build_all(for_email=False)
        + "</body></html>"
    )
    esc = nb_html.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        '<iframe srcdoc="' + esc + '" style="width:100%;border:none;min-height:900px;" '
        "onload=\"this.style.height=(this.contentDocument.body.scrollHeight+40)+'px'\"></iframe>"
    ))
    print("\u2713 Display done")

# ── Send Email ───────────────────────────────────────────────────────
if SEND_EMAIL:
    try:
        import pythoncom, win32com.client
        email_html = (
            "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
            "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
            "<div style='padding:20px 24px;background:#fff;" + FONT + "'>"
            + _greeting()
            + build_all(for_email=True)
            + _signature()
            + "</div>"
        )

        def send_auto(to, cc, subject, html_body, attachment=None, quit_after=True):
            pythoncom.CoInitialize()
            was_on = any(p.name().lower()=="outlook.exe" for p in psutil.process_iter(["name"]))
            if not was_on:
                for exe in [
                    r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                    r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
                ]:
                    if os.path.exists(exe): subprocess.Popen([exe]); break
                print("\u23f3 Starting Outlook...")
                for _ in range(30):
                    time.sleep(1)
                    try: win32com.client.GetActiveObject("Outlook.Application"); break
                    except: pass
            try:
                ol   = win32com.client.Dispatch("Outlook.Application")
                ol.GetNamespace("MAPI").Logon()
                mail = ol.CreateItem(0)
                mail.To=to; mail.CC=cc; mail.Subject=subject; mail.HTMLBody=html_body
                # Attach Excel file
                if attachment and os.path.exists(attachment):
                    mail.Attachments.Add(os.path.abspath(attachment))
                    print(f"\u2713 Attachment added: {os.path.basename(attachment)}")
                mail.Send()
                print(f"\u2713 Email sent \u2192 {to}")
                time.sleep(3)
                # Delete temp attachment immediately after sending
                if attachment and os.path.exists(attachment):
                    try:
                        os.remove(attachment)
                        print(f"\u2713 Temp file deleted: {os.path.basename(attachment)}")
                    except Exception as del_err:
                        print(f"\u26a0\ufe0f Could not delete temp file: {del_err}")
            finally:
                if quit_after and not was_on:
                    try: ol.Quit(); print("\u2713 Outlook closed")
                    except: pass

        send_auto(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, email_html,
                  attachment=att_path)

    except ImportError:
        print("\u274c win32com not available. Install: pip install pywin32")
    except Exception as e:
        print(f"\u274c Email error: {e}")
        import traceback; traceback.print_exc()


⏳ Computing tables...
✓ T1 MoM     : 10 rows | months: ['May-26', 'Jun-26', 'Jul-26', 'Aug-26']
✓ T2 7-Day   : 7 rows | dates : ['22-Aug', '23-Aug', '24-Aug', '25-Aug', '26-Aug', '27-Aug', '28-Aug']
✓ T3 D-1     : 7 rows
✓ T4 MTD     : 8 rows
✓ T5 Top-5   : 6 rows
⏳ Preparing attachment...
✓ Temp file ready: missed_hours_detail.xlsx (2,804 rows, Aug-26 MTD)
✓ Attachment : 2804 rows | C:\Users\HUUCHI~1.NGU\AppData\Local\Temp\missed_hours_detail.xlsx


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Supervisor Name,May-26Missed (hrs),Jun-26Missed (hrs),Jul-26Missed (hrs),Aug-26Missed (hrs),Grand Total
Ann,34.93,38.43,32.13,30.56,136.05
Chau Thien Kim,62.53,64.92,34.19,53.63,215.27
Mia Minh Le,68.05,0.00,1.53,1.78,71.36
Nguyen Thi Anh Thu,54.98,57.27,34.05,46.25,192.55
Tran Hoang My Anh,0.00,5.79,17.05,19.38,42.21
Tran Thao Uyen,66.01,44.05,23.72,11.61,145.39
Tran Thi Ngoc Bich,4.66,32.66,0.00,0.00,37.32
Tran Tran,20.31,3.25,0.00,0.00,23.57
Truong Thien Thanh Toan,35.22,49.74,54.55,32.04,171.54
Grand Total,346.69,296.11,197.22,195.25,1035.26


✓ Display done
✓ Attachment added: missed_hours_detail.xlsx
✓ Email sent → puneet.suneja@concentrix.com;kirpan.patar@concentrix.com;ML.HOC.Expedia.Hierarchy@concentrix.com
✓ Temp file deleted: missed_hours_detail.xlsx
